# Waste Right: material detection fine-tuning

SSDLite MobileNetV3, initialized from COCO weights, trained on official TACO annotations. Plastic, paper, cardboard, glass, metal and needs-review are the six detection labels. Food waste and specialist handling stay with Gemini/OCR and resident review.

The notebook resumes an interrupted run, selects the best checkpoint on validation mAP, and evaluates the held-out test split after training. A model checkpoint alone does not establish household accuracy or deployment readiness.

## Runtime
Select a GPU runtime before running. No paid upgrades or compute purchases are required by this notebook. Availability depends on your Colab account.

In [ ]:
import torch, torchvision
print("torch", torch.__version__, "torchvision", torchvision.__version__)
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU"
print(torch.cuda.get_device_name(0))

## Training files in Drive
The project folder holds the dataset in 500-image archives plus a small training bundle. Google asks for access when mounting Drive. The files stay private.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import zipfile
folder = Path("/content/drive/MyDrive/Waste Right Model Training")
archives = sorted(folder.glob("waste-right-*.zip"))
assert len(archives) >= 4, "Dataset archives and training bundle are missing"
workspace = Path("/content/waste-right")
workspace.mkdir(exist_ok=True)
for archive in archives:
    with zipfile.ZipFile(archive) as z:
        for name in z.namelist():
            assert (workspace / name).resolve().is_relative_to(workspace.resolve()), "Unsafe archive path"
        z.extractall(workspace)
print("Training package extracted")

In [ ]:
%pip install -q -r /content/waste-right/training/requirements.txt

## Dataset audit
Check image integrity, scaled bounding boxes, every-class coverage, and source/duplicate separation across train, validation and test.

In [ ]:
!python /content/waste-right/training/audit_dataset.py /content/waste-right/data

## Train and checkpoint
Thirty epochs at most, with early stopping after seven validation checks without improvement. Every epoch saves last.pt; best.pt uses validation mAP. Checkpoints and metrics are written to Drive. Re-run this cell after a disconnect to resume with the same data and settings.

In [ ]:
import subprocess, sys
output = folder / "runs" / "ssdlite-v1"
subprocess.run([sys.executable,"-u","/content/waste-right/training/train_detector.py","--data","/content/waste-right/data","--output",str(output),"--epochs","30","--batch-size","8","--workers","2","--resume","--require-cuda"], check=True)

## Results
Report mAP@0.50:0.95 and mAP@0.50, per-class values and the selected epoch. Do not describe smoke-test results as model accuracy. TACO contains outdoor litter and cannot establish performance on household camera views.

In [ ]:
import json
report = json.loads((output / "evaluation.json").read_text())
print(json.dumps(report["test"], indent=2))
print("Selected epoch:", report["selected_epoch"])
print((output / "model_card.json").read_text())

## Sources and intended use
- TACO: https://github.com/pedropro/TACO
- Dataset release and CC BY 4.0 attribution: https://zenodo.org/records/3587843
- Model: https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.detection.ssdlite320_mobilenet_v3_large.html

This model proposes visible material labels and boxes. It cannot determine weight, contamination hidden from view, local disposal rules or whether an item is safe to collect. People retain the final decision. Independent household evaluation and confidence-threshold calibration remain required before app integration.